In [ ]:
import requests
import pandas as pd
import re
import plotly.express as px
from fuzzywuzzy import fuzz
from fuzzywuzzy import process
import folium


In [ ]:
#  Data Acquisition
def fetch_openaq_data(city, parameter):
    url = f"https://api.openaq.org/v2/measurements?city={city}&parameter={parameter}&limit=10000"
    response = requests.get(url)
    if response.status_code == 200:
        data = response.json()['results']
        return pd.DataFrame(data)
    else:
        print(f"Error fetching data for {city} and {parameter}: {response.status_code}")
        return None

cities = ["London", "Paris", "New York", "Tokyo", "Beijing"]
parameters = ["pm25", "pm10", "o3"]

all_data = []
for city in cities:
    for parameter in parameters:
        df = fetch_openaq_data(city, parameter)
        if df is not None:
            df['city'] = city
            df['parameter'] = parameter
            all_data.append(df)

df = pd.concat(all_data, ignore_index=True)

In [ ]:
#  Data Cleaning and Standardization (Regexp & FuzzyWuzzy)
def clean_location(location):
    location = re.sub(r"[^a-zA-Z0-9\s]", "", location)
    location = location.strip()
    return location

df['location'] = df['location'].apply(clean_location)

def standardize_city_names(city_series):
    unique_cities = list(city_series.unique())
    standardized_cities = {}
    for city in city_series:
        match, score = process.extractOne(city, unique_cities, scorer=fuzz.ratio)
        if score >= 80: # Adjust threshold as needed
            standardized_cities[city] = match
        else:
            standardized_cities[city] = city # keep original if no good match
    return city_series.map(standardized_cities)

df['location'] = standardize_city_names(df['location'])

In [ ]:
#  Reshape Data (Melt)
df_melted = pd.melt(df, id_vars=['city', 'location', 'date.utc', 'parameter', 'unit'],
                     value_vars=['value'], var_name='measurement', value_name='measurement_value')

In [ ]:
#  Pivot Table (Pivot)
df_pivot = pd.pivot_table(df_melted, values='measurement_value', index=['city', 'date.utc'],
                           columns='parameter', aggfunc='mean')

In [ ]:
#  Stack/Unstack
df_stacked = df_pivot.stack().reset_index(name='value')

In [ ]:
#  Map/Recode
parameter_mapping = {
    'pm25': 'PM 2.5',
    'pm10': 'PM 10',
    'o3': 'Ozone'
}
df_stacked['parameter'] = df_stacked['parameter'].map(parameter_mapping)

In [ ]:
#  Descriptive Statistics
print("\nDescriptive Statistics:\n", df_stacked['value'].describe())

In [ ]:
#  Plotly Graphs
fig_line = px.line(df_stacked, x='date.utc', y='value', color='city', facet_col='parameter',
                   title='Air Quality Trends Over Time')
fig_line.show()


In [ ]:
#  Folium Map
city_locations = {
    "London": [51.5074, -0.1278],
    "Paris": [48.8566, 2.3522],
    "New York": [40.7128, -74.0060],
    "Tokyo": [35.6895, 139.6917],
    "Beijing": [39.9042, 116.4074]
}

m = folium.Map(location=[0, 0], zoom_start=2)

for city, location in city_locations.items():
    city_data = df[df['city'] == city].groupby('parameter')['value'].mean().to_dict()
    popup_text = f"<b>{city}</b><br>"
    for param, value in city_data.items():
        popup_text += f"{param}: {value:.2f} {df['unit'].iloc[0]}<br>"

    folium.Marker(location, popup=popup_text).add_to(m)

m.save("air_quality_map.html")
print("\nFolium map saved to air_quality_map.html")